# PASCAL VOC Object Detection 

In [1]:
import os, sys, glob, random, shutil, importlib, argparse, copy, pickle
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets import STL10, ImageFolder
from torchvision.transforms import ToTensor, functional as TF
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from ultralytics import YOLO
from IPython.display import Image, display
import kagglehub, yaml as pyyaml
import xml.etree.ElementTree as ET

sys.path.append("/users/jmatthia/deep_learning/code/assignment_3/")
import correct_split; importlib.reload(correct_split)
from correct_split import assert_clean_yolo_split

random.seed(42)
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())

cuda: True gpus: 1


Adding weights and biases key to track training

In [2]:
# login for weights and biases 
wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /users/jmatthia/.netrc
wandb: Currently logged in as: jan-matthias (jan-matthias-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Data import and correct format for YOLOv8

In [3]:
src = kagglehub.dataset_download("huanghanchina/pascal-voc-2012")
dst = "/users/jmatthia/deep_learning/data/pascal_voc_2012_v2"

print("SRC:", src)
print("SRC exists:", os.path.exists(src))

print("DST:", dst)
os.makedirs(dst, exist_ok=True)
print("DST exists:", os.path.exists(dst))

print("Copying...")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("Copied.")

voc = os.path.join(dst, "VOCdevkit", "VOC2012")
print("VOC path:", voc)
print("VOC exists:", os.path.exists(voc))
if os.path.exists(voc):
    print("VOC contains:", sorted(os.listdir(voc))[:10])

SRC: /users/jmatthia/.cache/kagglehub/datasets/huanghanchina/pascal-voc-2012/versions/1
SRC exists: True
DST: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2
DST exists: True
Copying...
Copied.
VOC path: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOCdevkit/VOC2012
VOC exists: False


Check if paths and files are correct (was having some issues with this during the creation of the dataset)

In [4]:
BASE = "/users/jmatthia/deep_learning/data/pascal_voc_2012_v2/"  

print("Scanning:", BASE)
for d in sorted(os.listdir(BASE)):
    p = os.path.join(BASE, d)
    if os.path.isdir(p):
        print("\nChecking:", p)
        print("Contains:", sorted(os.listdir(p))[:10])
        has = all(os.path.isdir(os.path.join(p, x)) for x in ["JPEGImages", "Annotations", "ImageSets"])
        print("→ Valid VOC root:", has)

Scanning: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2/

Checking: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012
Contains: ['Annotations', 'ImageSets', 'JPEGImages', 'SegmentationClass', 'SegmentationObject', 'VOC_split']
→ Valid VOC root: True

Checking: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2/voc2012
Contains: ['VOC2012']
→ Valid VOC root: False


We need to change the format of the images and annotations for it to work with YOLO. Here we are splitting the dataset into train (80%), validation (10%) and test (10%). Here we are only writing train.txt, val.txt and test.txt.

In [5]:
VOC_ROOT = "/users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012"
IMG_DIR  = os.path.join(VOC_ROOT, "JPEGImages")
SPLIT_DIR = os.path.join(VOC_ROOT, "ImageSets", "Main")
os.makedirs(SPLIT_DIR, exist_ok=True)
# get image ids (no extension)
ids = sorted(os.path.splitext(f)[0] for f in os.listdir(IMG_DIR) if f.endswith(".jpg"))
random.shuffle(ids)

n = len(ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_ids = ids[:n_train]
val_ids   = ids[n_train:n_train+n_val]
test_ids  = ids[n_train+n_val:]

def write_txt(filename, id_list):
    with open(os.path.join(SPLIT_DIR, filename), "w") as f:
        f.write("\n".join(id_list))
# creating the split
write_txt("train.txt", train_ids)
write_txt("val.txt", val_ids)
write_txt("test.txt", test_ids)

print("Total:", n)
print("Train:", len(train_ids))
print("Val:", len(val_ids))
print("Test:", len(test_ids))

Total: 17125
Train: 13700
Val: 1712
Test: 1713


YOLO expects a yaml file with the path to the train, val and test sets. Additionally it should also contain the various classes of the images 

In [6]:
VOC_SPLIT = "/users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012/VOC_split"
os.makedirs(VOC_SPLIT, exist_ok=True)
yaml_path = os.path.join(VOC_SPLIT, "voc.yaml")

In [7]:
yaml_text = f"""path: {VOC_SPLIT}
train: images/train
val: images/val
test: images/test
names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""
with open(yaml_path, "w") as f:
    f.write(yaml_text)
print("Wrote:", yaml_path, "Exists:", os.path.exists(yaml_path))

Wrote: /users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012/VOC_split/voc.yaml Exists: True


Function from YOLOv5 to convert images into the correct format 

In [8]:
# ---- function directly from YOLOv5 
def convert_label(path, lb_path, image_id,yaml):
      def convert_box(size, box):
          dw, dh = 1. / size[0], 1. / size[1]
          x, y, w, h = (box[0] + box[1]) / 2.0 - 1, (box[2] + box[3]) / 2.0 - 1, box[1] - box[0], box[3] - box[2]
          return x * dw, y * dh, w * dw, h * dh

      in_file = open(path / "Annotations" / f"{image_id}.xml")
      out_file = open(lb_path, 'w')
      tree = ET.parse(in_file)
      root = tree.getroot()
      size = root.find('size')
      w = int(size.find('width').text)
      h = int(size.find('height').text)

      names = list(yaml['names'].values())  # names list
      for obj in root.iter('object'):
          cls = obj.find('name').text
          if cls in names and int(obj.find('difficult').text) != 1:
              xmlbox = obj.find('bndbox')
              bb = convert_box((w, h), [float(xmlbox.find(x).text) for x in ('xmin', 'xmax', 'ymin', 'ymax')])
              cls_id = names.index(cls)  # class id
              out_file.write(" ".join([str(a) for a in (cls_id, *bb)]) + '\n')

Converting the images into the correct format 

In [9]:
VOC2012   = Path("/users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012")
IMG_DIR   = VOC2012 / "JPEGImages"
SPLIT_DIR = VOC2012 / "ImageSets" / "Main"
VOC_SPLIT = VOC2012 / "VOC_split"

yaml = pyyaml.safe_load(open(VOC_SPLIT / "voc.yaml"))

# folders + materialize
for s in ["train","val","test"]:
    (VOC_SPLIT/"images"/s).mkdir(parents=True, exist_ok=True)
    (VOC_SPLIT/"labels"/s).mkdir(parents=True, exist_ok=True)

for split in ["train","val","test"]:
    ids = (SPLIT_DIR / f"{split}.txt").read_text().split()
    for image_id in ids:
        shutil.copy2(IMG_DIR / f"{image_id}.jpg", VOC_SPLIT/"images"/split/f"{image_id}.jpg")
        convert_label(VOC2012,
                      VOC_SPLIT/"labels"/split/f"{image_id}.txt",
                      image_id,yaml)

Quick check to see if the changed data format is what we expect. Additionally I also check if the image size is correct 

In [10]:
VOC_SPLIT = VOC2012 / "VOC_split"
def ok_yolo(line):
    try:
        c,x,y,w,h = line.split()
        c=int(c); x,y,w,h=map(float,(x,y,w,h))
        return (c>=0 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1 and
                0<=x-w/2<=1 and 0<=x+w/2<=1 and
                0<=y-h/2<=1 and 0<=y+h/2<=1)
    except: return False
for split in ["train","val","test"]:
    imgs = glob.glob(f"{VOC_SPLIT}/images/{split}/*.jpg")
    lbls = glob.glob(f"{VOC_SPLIT}/labels/{split}/*.txt")
    print(split, "images:", len(imgs), "labels:", len(lbls))
    # image-label count match
    print("count match:", len(imgs)==len(lbls))
    # sample format check
    if lbls:
        bad = 0
        with open(lbls[0]) as f:
            for line in f:
                if line.strip() and not ok_yolo(line.strip()):
                    bad += 1
        print("format ok:", bad==0)
    print()

train images: 13700 labels: 13700
count match: True
format ok: True

val images: 1712 labels: 1712
count match: True
format ok: True

test images: 1713 labels: 1713
count match: True
format ok: True



Check that the split is correct and there is no data leakage (had issues with this before)

In [11]:
assert_clean_yolo_split("/users/jmatthia/deep_learning/data/pascal_voc_2012_v2/VOC2012/VOC_split")

✓ YOLO split OK (filename-level): {'train': 13700, 'val': 1712, 'test': 1713}


# YOLOv5 training + logging with weights and biases 

Here I used a patience of 3, after three consecutive rises in the validation score the model stops training. In regards to other hyperparameters I utilized YOLOv5's defult ones. Optimizer: SGD, Scheduler: Cosine LR. 

%cd /users/jmatthia/deep_learning/code/assignment_3/yolov5

#running the model
!python train.py \
  --img 640 \
  --batch 4 \
  --epochs 200 \
  --patience 20 \
  --data /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml \
  --cfg models/yolov5m.yaml \
  --weights yolov5m.pt \
  --project runs/train \
  --name yolov5m_voc_pretrained \
  --save-period 1


Here we train YOLOv5 from scratch (complete outputs are in the .out file and were submitted to the cluster via a sbatch file)

%cd /users/jmatthia/deep_learning/code/assignment_3/yolov5

! python train.py \
  --img 640 \
  --batch-size -1 \
  --epochs 200 \
  --patience 50 \
  --data /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml \
  --cfg models/yolov5m.yaml \
  --weights '' \
  --project runs/train \
  --name yolov5m_voc_from_scratch_test \
  --save-period 1

# Part 3: questions from the assignment 

**a) Anchor boxes/priors**

YOLOv5 is a single-stage object detector that uses predefined anchor boxes to guide bounding box regression. YOLOv1 divides the image into a single SxS grid, whereas YOLOv5 predicts objects at multiple feature scales (typically corresponds to small, medium and large). At each scale, every spatial location is associated with three anchor boxes of different sizes and aspect ratios. For each anchor at each grid location, YOLOv5 predicts offset for the bounding box center (x,y) and dimensions (w,h). Additionally it also predicts an objectness score and class probability. The model does not predict absolute coordinates, but rather transformations relative to the anchor box dimensions and grid cell location. 


In contrast R-CNN, a two-stage approach. First, a Region Proposal algorithm which generated candidate objects regions using anchor boxes. In a second stage a linear regression model receives the region proposal, and predicts the off-set from these coordinates.


**b) Region Proposal**

Instead of generating candidate regions first (as in Faster R-CNN), YOLOv5 performs detection at multiple feature map scales using a feature pyramid structure. It processes the images through its backbone (CSP-based architecture) and feeds the features into the following module which combines high-level semantic information with lower-level spatial information through both bottom-up and top-down pathways. This produces multiple feature maps at different spatial resolutions. Each spatial resolution has an individual detection head that predicts the bounding boxes, objectiveness and class probability scores. This architecture allows YOLOv5 to predict without the need of a separate proposal stage. 


In [12]:
# get the image
# image from https://github.com/ultralytics/yolov5/issues/280
Image(url="/users/jmatthia/deep_learning/code/assignment_3/Screenshot 2026-02-25 at 10.19.07.png", width=600, height=600)

**c) Bounding Box regression**

YOLOv5 performs bounding box regression by predicting the coordinate offset relative to the anchor box and grid cell locations. For each anchor at each grid cell, it predicts the offset from the box center (x,y) and the dimension (w,h). During training, the predicted boxes are matched to the ground truth boxes. Older YOLO versions used a sum of squares to calculate the coordinate loss. YOLOv5 utilizes a Complete Intersection over Union loss (CloU). CloU, does not only consider overlap area, but also the center distance aspect ratio consistency which in turn leads to a more stable and accurate box refinement. 

**d) Classifications vs Localisation**

Within each detection head (corresponding to small, medium or large images), YOLOv5 predicts localisation and classification. Here the bounding box coordinates offset, objectness score and class probabilities are predicted. In a subsequent stage the box coordinates are combined with the box coordinates. The objectness score represents the model’s confidence that an object is present within that predicted bounding box. In parallel, the model predicts class probabilities for each object category.During the testing phase, the final confidence score is the product of the objectiveness score and predicted class probability. YOLO predicts all of this, from the same detection head with one final tensor. 


Below is part of the code from yolo.py from the YOLOv5 Detect class that is used to decode the tensor into the bounding box coordinates, objectness and class probabilities

In [13]:
def forward(self, x):
        """Processes input through YOLOv5 layers, altering shape for detection: `x(bs, 3, ny, nx, 85)`."""
        z = []  # inference output
        for i in range(self.nl):
            x[i] = self.m[i](x[i])  # conv
            bs, _, ny, nx = x[i].shape  # x(bs,255,20,20) to x(bs,3,20,20,85)
            x[i] = x[i].view(bs, self.na, self.no, ny, nx).permute(0, 1, 3, 4, 2).contiguous()

            if not self.training:  # inference
                if self.dynamic or self.grid[i].shape[2:4] != x[i].shape[2:4]:
                    self.grid[i], self.anchor_grid[i] = self._make_grid(nx, ny, i)

                if isinstance(self, Segment):  # (boxes + masks)
                    xy, wh, conf, mask = x[i].split((2, 2, self.nc + 1, self.no - self.nc - 5), 4)
                    xy = (xy.sigmoid() * 2 + self.grid[i]) * self.stride[i]  # xy
                    wh = (wh.sigmoid() * 2) ** 2 * self.anchor_grid[i]  # wh
                    y = torch.cat((xy, wh, conf.sigmoid(), mask), 4)
                else:  # Detect (boxes only)
                    xy, wh, conf = x[i].sigmoid().split((2, 2, self.nc + 1), 4)
                    xy = (xy * 2 + self.grid[i]) * self.stride[i]  # xy
                    wh = (wh * 2) ** 2 * self.anchor_grid[i]  # wh
                    y = torch.cat((xy, wh, conf), 4)
                z.append(y.view(bs, self.na * nx * ny, self.no))

        return x if self.training else (torch.cat(z, 1),) if self.export else (torch.cat(z, 1), x)

**e) Confidence Thresholding & Non-Maximum Suppression (NMS): Show how your detector filters out overlapping or low-confidence predictions using confidence thresholds and NMS.**


YOLO outputs multiple candidate boxes and all candidate boxes have box coordinates, objectness score and class probabilities. During inference, YOLOv5 first computes the per-class confidence score (score = p(obj) x p(c given obj), to which a threshold is applied (C1). Here YOLOv5 is already removing low-confidence candidates. This reduces the number of boxes substantially but does not eliminate duplicates. To solve this YOLOv5 utilizes non-maximum suppression (NMS) utilizing an IoU threshold (C2). NMS keeps the highest scoring detection and suppresses other boxes that overlap it too much. The result from this, are boxes that contain non-overlapping bounding boxes.


For YOLO NMS is done during inference, more specifically in this function from the general.py code.


In [14]:
def non_max_suppression(
   prediction,
   conf_thres=0.25,
   iou_thres=0.45,
   classes=None,
   agnostic=False,
   multi_label=False,
   labels=(),
   max_det=300,
   nm=0,  # number of masks
):
   """Non-Maximum Suppression (NMS) on inference results to reject overlapping detections.


   Returns:
       list of detections, on (n,6) tensor per image [xyxy, conf, cls]
   """


**F) Evaluation Metrics**

Intersection over the union (IoU) is used to measure the overlap between a predicted bounding box and the ground-truth. It is defined as the Area of Intersection / Area of the union. An IoU of 1 indicates a perfect overlap, whereas 0 indicates no overlap. During the evaluation step, boxes are considered a True Positive (TP) if its IoU exceeds a predefined threshold, and the class is also correct. Otherwise it is considered a FP. 

In the case of YOLO this is computed within the bbox_iou function in the metrics.py. The below function is where YOLO calculates the the CIoU. CloU, does not only consider overlap area, but also the center distance aspect ratio consistency which in turn leads to a more stable and accurate box refinement. 

In [15]:
def bbox_iou(box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7):
    """Calculates IoU, GIoU, DIoU, or CIoU between two boxes, supporting xywh/xyxy formats.

    Input shapes are box1(1,4) to box2(n,4).
    """
    # Get the coordinates of bounding boxes
    if xywh:  # transform from xywh to xyxy
        (x1, y1, w1, h1), (x2, y2, w2, h2) = box1.chunk(4, -1), box2.chunk(4, -1)
        w1_, h1_, w2_, h2_ = w1 / 2, h1 / 2, w2 / 2, h2 / 2
        b1_x1, b1_x2, b1_y1, b1_y2 = x1 - w1_, x1 + w1_, y1 - h1_, y1 + h1_
        b2_x1, b2_x2, b2_y1, b2_y2 = x2 - w2_, x2 + w2_, y2 - h2_, y2 + h2_
    else:  # x1, y1, x2, y2 = box1
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
        w1, h1 = b1_x2 - b1_x1, (b1_y2 - b1_y1).clamp(eps)
        w2, h2 = b2_x2 - b2_x1, (b2_y2 - b2_y1).clamp(eps)

    # Intersection area
    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp(0) * (
        b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)
    ).clamp(0)

    # Union Area
    union = w1 * h1 + w2 * h2 - inter + eps

The mean average precision (mAP) evaluates object detection performance by averaging the precision-recall across different thresholds and classes. Precision is calculated by dividing TP by TP + FP. TP and FP are defined by a specific IoU threshold. Recall is calculated as TP/TP + FN. It is a tradeoff between the two, higher precision for lower recall.

For a fixed IoU threshold (e.g., 0.5), predictions are first sorted in descending order according to their confidence score (objectness × class probability). A prediction is counted as a True Positive (TP) if it correctly matches a ground-truth box with IoU greater than or equal to the threshold and has the correct class label. Otherwise, it is counted as a False Positive (FP). Ground-truth boxes that are not matched by any prediction are counted as False Negatives (FN). For mAP 0.05-0.95, the IoU threshold changes and it corresponds to the average of all different thresholds and all different classes (in our case it would be 20 classes).


Below is part of the YOLOv5's AP per class function 

In [16]:
def ap_per_class(tp, conf, pred_cls, target_cls, plot=False, save_dir=".", names=(), eps=1e-16, prefix=""):
    """Compute the average precision, given the recall and precision curves.

    Source: https://github.com/rafaelpadilla/Object-Detection-Metrics.

    Args:
        tp: True positives (nparray, nx1 or nx10).
        conf: Objectness value from 0-1 (nparray).
        pred_cls: Predicted object classes (nparray).
        target_cls: True object classes (nparray).
        plot: Plot precision-recall curve at mAP@0.5
        save_dir: Plot save directory

    Returns:
        The average precision as computed in py-faster-rcnn.
    """
    # Sort by objectness
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]

    # Find unique classes
    unique_classes, nt = np.unique(target_cls, return_counts=True)
    nc = unique_classes.shape[0]  # number of classes, number of detections

    # Create Precision-Recall curve and compute AP for each class
    px, py = np.linspace(0, 1, 1000), []  # for plotting
    ap, p, r = np.zeros((nc, tp.shape[1])), np.zeros((nc, 1000)), np.zeros((nc, 1000))
    


# Test on held-out test split

As a first step, we are calculating the scores on the pre-trained model with coco

Running the model on our test set

In [17]:
%cd /users/jmatthia/deep_learning/code/assignment_3/yolov5

!python val.py \
  --weights /users/jmatthia/deep_learning/code/assignment_3/yolov5/runs/train/yolov5m_voc_coco_pretrained/weights/best.pt \
  --data /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml \
  --task test \
  --img 640 \
  --save-json 

/users/jmatthia/deep_learning/code/assignment_3/yolov5


/users/jmatthia/deep_learning/env/yolov5_cu121/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


val: data=/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml, weights=['/users/jmatthia/deep_learning/code/assignment_3/yolov5/runs/train/yolov5m_voc_coco_pretrained/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=True, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.10.8 torch-2.5.1 CUDA:0 (NVIDIA A100 80GB PCIe, 81158MiB)

Fusing layers... 
YOLOv5m summary: 212 layers, 20929713 parameters, 0 gradients, 48.1 GFLOPs
test: Scanning /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_sp
                 Class     Images  Instances          P          R      mAP50   
                   all       1713       3559      0.752      0.686      0.712      0.534
             aeroplane       1713        102      0.929      0.

Checking YOLOv5 trained by scratch

In [18]:
%cd /users/jmatthia/deep_learning/code/assignment_3/yolov5

!python val.py \
  --weights /users/jmatthia/deep_learning/code/assignment_3/yolov5/runs/train/yolov5m_voc_from_scratch/weights/best.pt \
  --data /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml \
  --task test \
  --img 640 \
  --save-json 

/users/jmatthia/deep_learning/code/assignment_3/yolov5
val: data=/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml, weights=['/users/jmatthia/deep_learning/code/assignment_3/yolov5/runs/train/yolov5m_voc_from_scratch/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=True, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.10.8 torch-2.5.1 CUDA:0 (NVIDIA A100 80GB PCIe, 81158MiB)

Fusing layers... 
YOLOv5m summary: 212 layers, 20929713 parameters, 0 gradients, 48.1 GFLOPs
test: Scanning /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_sp
                 Class     Images  Instances          P          R      mAP50   
                   all       1713       3559      0.712      0.631      0.663      0.472
           

Sample image from the train set

In [19]:
Image(url="/users/jmatthia/deep_learning/code/assignment_3/yolov5/runs/val/exp2/val_batch2_labels.jpg", width=600, height=600)